# Domain-Specific Text Generation for Machine Translation
## A resource-constrained reproduction and filtering ablation (Arabic to English)
**Student:** Nguyễn Hoàng Anh Kiệt | **ID:** DE200278 | **Class:** AI20B

Based on Moslem et al. (AMTA 2022). Repository: https://github.com/Anhkietttt/TMG301

This notebook is self-contained: its source modules are embedded below, so no GitHub login, repository clone, manually uploaded data, or API key is needed. Enable **GPU and Internet**, start a fresh Kaggle session, and **Run All**. It downloads public models and datasets and performs real generation, back-translation, training and evaluation. There are no prefilled experimental scores. Keep `/kaggle/working/domain_mt_runs` when resuming within the same environment. Across sessions, copy a saved run folder from an attached Kaggle output into that directory before running; do not overwrite it with a different configuration.

**Scope:** only Setup 1, Arabic-to-English. Pythia-410M replaces GPT-J-6B; pretrained Marian checkpoints replace the paper's trained Transformer-Big baselines. We use small, document-disjoint TICO subsets and OPUS-100 replay, not the original 40-million-pair training recipe. This is a method reproduction, NOT an exact score reproduction. All choices are frozen before final testing.

In [ ]:
%pip install -q "transformers==4.57.6" "datasets>=4.3,<5" "accelerate>=1.11,<2" "sacrebleu>=2.4,<3" "sentencepiece>=0.2,<0.3" "sacremoses>=0.1,<0.2" "sentence-transformers>=3.4,<6" "langid==1.1.6" "pandas>=2.2,<4" "matplotlib>=3.8,<4" "scikit-learn>=1.5,<2" "nbformat>=5.10,<6"


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json
import sys
from pathlib import Path
EMBEDDED_FILES = json.loads('{"src/domainmt/__init__.py": "\\"\\"\\"Resource-constrained domain adaptation experiments.\\"\\"\\"\\n", "src/domainmt/data.py": "\\"\\"\\"Pure-Python data validation and document-disjoint partitions.\\"\\"\\"\\nimport csv\\nimport hashlib\\nimport io\\nimport random\\nimport re\\nimport unicodedata\\nimport zipfile\\n\\n\\ndef normalize(text):\\n    return \' \'.join(unicodedata.normalize(\'NFKC\', text).casefold().split())\\n\\n\\ndef text_hash(text):\\n    return hashlib.sha256(normalize(text).encode(\'utf-8\')).hexdigest()\\n\\n\\ndef is_arabic(text):\\n    letters = [c for c in text if c.isalpha()]\\n    return bool(letters) and sum(\'\\\\u0600\' <= c <= \'\\\\u06ff\' for c in letters) / len(letters) >= 0.5\\n\\n\\ndef clean_pairs(rows):\\n    result, seen = [], set()\\n    for row in rows:\\n        ar, en = row.get(\'ar\'), row.get(\'en\')\\n        if not isinstance(ar, str) or not isinstance(en, str):\\n            continue\\n        key = (normalize(ar), normalize(en))\\n        if not all(key) or key in seen or key[0] == key[1]:\\n            continue\\n        if not is_arabic(ar) or is_arabic(en):\\n            continue\\n        seen.add(key)\\n        result.append(dict(row))\\n    return result\\n\\n\\ndef read_tico(path):\\n    rows = []\\n    with zipfile.ZipFile(path) as archive:\\n        for split in (\'dev\', \'test\'):\\n            name = f\'tico19-testset/{split}/{split}.en-ar.tsv\'\\n            reader = csv.DictReader(io.StringIO(archive.read(name).decode(\'utf-8-sig\')), delimiter=\'\\\\t\')\\n            for row in reader:\\n                if row[\'sourceLang\'] != \'en\' or row[\'targetLang\'] != \'ar\':\\n                    raise ValueError(\'Unexpected TICO direction\')\\n                rows.append({\'ar\': row[\'targetString\'], \'en\': row[\'sourceString\'],\\n                             \'id\': row[\'stringID\'], \'document\': row[\'stringID\'].split(\':\')[0],\\n                             \'license\': row[\'license\'], \'official_split\': split})\\n    return clean_pairs(rows)\\n\\n\\ndef split_documents(rows, seed=42):\\n    documents = sorted({r[\'document\'] for r in rows})\\n    random.Random(seed).shuffle(documents)\\n    n = max(1, round(len(documents) * 0.2))\\n    groups = {\'test\': set(documents[:n]), \'dev\': set(documents[n:2*n]), \'train\': set(documents[2*n:])}\\n    splits = {name: [r for r in rows if r[\'document\'] in docs] for name, docs in groups.items()}\\n    # Reserve evaluation strings first, removing cross-partition exact overlap.\\n    forbidden_ar, forbidden_en = set(), set()\\n    for name in (\'test\', \'dev\', \'train\'):\\n        splits[name] = [r for r in splits[name] if text_hash(r[\'ar\']) not in forbidden_ar\\n                        and text_hash(r[\'en\']) not in forbidden_en]\\n        forbidden_ar.update(text_hash(r[\'ar\']) for r in splits[name])\\n        forbidden_en.update(text_hash(r[\'en\']) for r in splits[name])\\n    return splits\\n\\n\\ndef assert_disjoint(splits):\\n    names = list(splits)\\n    for i, left in enumerate(names):\\n        for right in names[i+1:]:\\n            for language in (\'ar\', \'en\'):\\n                a = {text_hash(r[language]) for r in splits[left]}\\n                b = {text_hash(r[language]) for r in splits[right]}\\n                if a & b:\\n                    raise AssertionError(f\'{left}/{right}: overlapping {language} text\')\\n\\n\\ndef continuation_sentences(text):\\n    # Keep only complete punctuation-terminated sentences, not the trailing fragment.\\n    return [s.strip() for s in re.findall(r\'[^.!?\\\\n]+[.!?](?:[\\"\\\\u201d\\\\u2019])?\', text)\\n            if 6 <= len(s.split()) <= 80 and not re.search(r\'https?://|<[^>]+>\', s)]\\n\\n\\ndef token_jaccard(a, b):\\n    a, b = set(normalize(a).split()), set(normalize(b).split())\\n    return len(a & b) / max(1, len(a | b))\\n", "src/domainmt/experiment.py": "\\"\\"\\"Staged, resumable Kaggle experiment; no fabricated scores or silent fallbacks.\\"\\"\\"\\nimport gc\\nimport hashlib\\nimport importlib.metadata\\nimport json\\nimport math\\nimport random\\nimport time\\nimport urllib.request\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom datasets import Dataset, load_dataset\\nfrom sacrebleu.metrics import BLEU, CHRF, TER\\nfrom transformers import (AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer,\\n                          DataCollatorForSeq2Seq, Seq2SeqTrainer,\\n                          Seq2SeqTrainingArguments, set_seed)\\n\\nfrom .data import (assert_disjoint, clean_pairs, continuation_sentences, is_arabic,\\n                   normalize, read_tico, split_documents, text_hash, token_jaccard)\\n\\n\\ndef write_json(path, value):\\n    path = Path(path)\\n    temp = path.with_suffix(\'.tmp\')\\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding=\'utf-8\')\\n    temp.replace(path)\\n\\n\\ndef free_memory():\\n    gc.collect()\\n    if torch.cuda.is_available():\\n        torch.cuda.empty_cache()\\n\\n\\nclass Experiment:\\n    def __init__(self, config, output_root):\\n        self.cfg = dict(config)\\n        if config[\'train_steps\'] % config[\'eval_steps\']:\\n            raise ValueError(\'train_steps must be divisible by eval_steps for final checkpoint evaluation.\')\\n        if not 0 < config[\'in_domain_probability\'] < 1:\\n            raise ValueError(\'Mixed sampling requires a probability strictly between zero and one.\')\\n        if not 0 < config[\'quality_keep_fraction\'] <= 0.5:\\n            raise ValueError(\'Matched ranking ablation keeps at most half the eligible pool.\')\\n        if config[\'prompt_count\'] > config[\'authentic_pairs\']:\\n            raise ValueError(\'prompt_count cannot exceed the authentic training pool.\')\\n        if not torch.cuda.is_available():\\n            raise RuntimeError(\'Enable a Kaggle GPU in Settings and restart. This training notebook requires CUDA.\')\\n        self.device = \'cuda:0\'\\n        self.seed = config[\'seed\']\\n        set_seed(self.seed)\\n        self.versions = {p: importlib.metadata.version(p) for p in\\n                         [\'torch\', \'transformers\', \'datasets\', \'sacrebleu\', \'sentence-transformers\']}\\n        code_digest = hashlib.sha256(Path(__file__).read_bytes() + Path(__file__).with_name(\'data.py\').read_bytes()).hexdigest()\\n        identity = {\'config\': config, \'versions\': self.versions, \'code_digest\': code_digest}\\n        run_id = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:12]\\n        self.root = Path(output_root) / run_id\\n        self.root.mkdir(parents=True, exist_ok=True)\\n        write_json(self.root / \'run_config.json\', identity)\\n        revisions = self.root / \'model_revisions.json\'\\n        self.model_revisions = json.loads(revisions.read_text()) if revisions.exists() else {}\\n        print(\'Run:\', self.root, \'| GPU:\', torch.cuda.get_device_name(0), flush=True)\\n\\n    def _record_model(self, name, model):\\n        self.model_revisions[name] = getattr(model.config, \'_commit_hash\', None)\\n        write_json(self.root / \'model_revisions.json\', self.model_revisions)\\n\\n    def _subset(self, rows, n):\\n        if len(rows) < n:\\n            raise ValueError(f\'Need {n} rows, only {len(rows)} remain. Review counts; do not silently change the protocol.\')\\n        return random.Random(self.seed).sample(rows, n)\\n\\n    def prepare_data(self):\\n        cache = self.root / \'prepared_data.json\'\\n        if cache.exists():\\n            self.data = json.loads(cache.read_text())\\n            assert_disjoint(self.data)\\n            return self.data_summary()\\n        archive = self.root / \'tico19-testset.zip\'\\n        if not archive.exists():\\n            urllib.request.urlretrieve(self.cfg[\'tico_url\'], archive)\\n        tico = read_tico(archive)\\n        partitions = split_documents(tico, self.seed)\\n        data = {f\'domain_{key}\': self._subset(partitions[key], self.cfg[size])\\n                for key, size in [(\'train\', \'authentic_pairs\'), (\'dev\', \'domain_dev\'), (\'test\', \'domain_test\')]}\\n        assert_disjoint(data)\\n        # All TICO strings are excluded from the general-domain pools, independent of outcomes.\\n        forbidden_ar = {text_hash(r[\'ar\']) for r in tico}\\n        forbidden_en = {text_hash(r[\'en\']) for r in tico}\\n        tokenizer = AutoTokenizer.from_pretrained(self.cfg[\'mt_model\'])\\n        for split, key in [(\'train\', \'generic_train\'), (\'validation\', \'generic_dev\'), (\'test\', \'generic_test\')]:\\n            print(\'Loading OPUS-100\', split, flush=True)\\n            stream = load_dataset(\'Helsinki-NLP/opus-100\', \'ar-en\', split=split, streaming=True)\\n            stream = stream.shuffle(seed=self.seed, buffer_size=5000)\\n            selected = []\\n            for i, item in enumerate(stream):\\n                pair = item[\'translation\']\\n                if not clean_pairs([pair]):\\n                    continue\\n                if text_hash(pair[\'ar\']) in forbidden_ar or text_hash(pair[\'en\']) in forbidden_en:\\n                    continue\\n                if min(len(pair[\'ar\'].split()), len(pair[\'en\'].split())) < 3:\\n                    continue\\n                if len(tokenizer(pair[\'ar\'])[\'input_ids\']) > self.cfg[\'max_length\']:\\n                    continue\\n                if len(tokenizer(text_target=pair[\'en\'])[\'input_ids\']) > self.cfg[\'max_length\']:\\n                    continue\\n                selected.append({\'ar\': pair[\'ar\'], \'en\': pair[\'en\'], \'id\': f\'opus100-{split}-{i}\'})\\n                forbidden_ar.add(text_hash(pair[\'ar\']))\\n                forbidden_en.add(text_hash(pair[\'en\']))\\n                if len(selected) == self.cfg[key]:\\n                    break\\n            if len(selected) != self.cfg[key]:\\n                raise RuntimeError(f\'{key}: not enough eligible examples ({len(selected)})\')\\n            data[key] = selected\\n        assert_disjoint(data)\\n        write_json(self.root / \'document_partitions.json\', {k: sorted({r[\'document\'] for r in v}) for k, v in partitions.items()})\\n        write_json(self.root / \'data_provenance.json\', {\\n            \'tico_sha256\': hashlib.sha256(archive.read_bytes()).hexdigest(),\\n            \'tico_rows_after_cleaning\': len(tico), \'generic\': \'Helsinki-NLP/opus-100 ar-en\',\\n            \'note\': \'Custom document-disjoint TICO partitions, not official benchmark or paper splits.\'})\\n        write_json(cache, data)\\n        self.data = data\\n        for name, rows in data.items():\\n            pd.DataFrame(rows).to_csv(self.root / f\'{name}.csv\', index=False)\\n        return self.data_summary()\\n\\n    def data_summary(self):\\n        return pd.DataFrame([{\'partition\': name, \'pairs\': len(rows)} for name, rows in self.data.items()])\\n\\n    def generate_targets(self):\\n        cache = self.root / \'generated_targets.json\'\\n        if cache.exists():\\n            self.targets = json.loads(cache.read_text())\\n            return pd.DataFrame(self.targets)\\n        import langid\\n        tokenizer = AutoTokenizer.from_pretrained(self.cfg[\'lm_model\'], padding_side=\'left\')\\n        tokenizer.pad_token = tokenizer.eos_token\\n        lm = AutoModelForCausalLM.from_pretrained(self.cfg[\'lm_model\'], torch_dtype=torch.float16).to(self.device).eval()\\n        self._record_model(self.cfg[\'lm_model\'], lm)\\n        prompts = self._subset(self.data[\'domain_train\'], self.cfg[\'prompt_count\'])\\n        blocked = [r[\'en\'] for name in (\'domain_train\', \'domain_dev\', \'domain_test\', \'generic_dev\', \'generic_test\')\\n                   for r in self.data[name]]\\n        blocked_hashes = {text_hash(t) for t in blocked}\\n        seen, targets, raw_records = set(), [], []\\n        resume = self.root / \'generation_progress.json\'\\n        if resume.exists():\\n            progress = json.loads(resume.read_text())\\n            raw_records = progress[\'raw\']\\n            targets = progress[\'targets\']\\n            seen = {text_hash(t[\'en\']) for t in targets}\\n        completed = len(raw_records)\\n        for index, row in enumerate(prompts):\\n            if index < completed:\\n                continue\\n            set_seed(self.seed + index)\\n            inputs = tokenizer(row[\'en\'].strip() + \'\\\\n\', return_tensors=\'pt\', truncation=True, max_length=128).to(self.device)\\n            with torch.inference_mode():\\n                outputs = lm.generate(**inputs, do_sample=True, top_k=50, top_p=0.95,\\n                                      max_new_tokens=self.cfg[\'lm_new_tokens\'],\\n                                      num_return_sequences=self.cfg[\'sequences_per_prompt\'],\\n                                      pad_token_id=tokenizer.eos_token_id)\\n            continuations = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)\\n            raw_records.append({\'prompt_id\': row[\'id\'], \'prompt\': row[\'en\'], \'continuations\': continuations})\\n            for text in continuations:\\n                for sentence in continuation_sentences(text):\\n                    key = text_hash(sentence)\\n                    if key in seen or key in blocked_hashes or langid.classify(sentence)[0] != \'en\':\\n                        continue\\n                    if any(token_jaccard(sentence, other) >= 0.8 for other in blocked):\\n                        continue\\n                    seen.add(key)\\n                    targets.append({\'en\': sentence, \'prompt_id\': row[\'id\'], \'id\': key[:20]})\\n            write_json(resume, {\'raw\': raw_records, \'targets\': targets})\\n            if (index + 1) % 10 == 0:\\n                print(f\'Prompts {index+1}/{len(prompts)}; accepted target sentences {len(targets)}\', flush=True)\\n        del lm\\n        free_memory()\\n        write_json(cache, targets)\\n        self.targets = targets\\n        return pd.DataFrame(targets)\\n\\n    def _translate(self, model, tokenizer, texts, beams=4):\\n        outputs = []\\n        model.eval()\\n        for start in range(0, len(texts), self.cfg[\'batch_size\']):\\n            inputs = tokenizer(texts[start:start+self.cfg[\'batch_size\']], return_tensors=\'pt\',\\n                               padding=True, truncation=True, max_length=self.cfg[\'max_length\']).to(self.device)\\n            with torch.inference_mode():\\n                predicted = model.generate(**inputs, num_beams=beams, do_sample=False,\\n                                           max_new_tokens=self.cfg[\'max_new_tokens\'])\\n            outputs.extend(tokenizer.batch_decode(predicted, skip_special_tokens=True))\\n        return outputs\\n\\n    def backtranslate_and_rank(self):\\n        cache = self.root / \'synthetic_pools.json\'\\n        if cache.exists():\\n            self.pools = json.loads(cache.read_text())\\n            return pd.DataFrame([{\'pool\': k, \'pairs\': len(v)} for k, v in self.pools.items()])\\n        if not self.targets:\\n            raise RuntimeError(\'No target sentences generated. Inspect generation_progress.json.\')\\n        raw_cache = self.root / \'backtranslations.json\'\\n        if raw_cache.exists():\\n            pairs = json.loads(raw_cache.read_text())\\n        else:\\n            tok = AutoTokenizer.from_pretrained(self.cfg[\'reverse_model\'])\\n            model = AutoModelForSeq2SeqLM.from_pretrained(self.cfg[\'reverse_model\']).to(self.device)\\n            self._record_model(self.cfg[\'reverse_model\'], model)\\n            ar = self._translate(model, tok, [r[\'en\'] for r in self.targets], beams=5)\\n            pairs = [{**r, \'ar\': text} for r, text in zip(self.targets, ar)]\\n            write_json(raw_cache, pairs)\\n            del model\\n            free_memory()\\n        pairs = clean_pairs(pairs)\\n        mt_tokenizer = AutoTokenizer.from_pretrained(self.cfg[\'mt_model\'])\\n        heldout = [r for key in (\'domain_dev\', \'domain_test\', \'generic_dev\', \'generic_test\') for r in self.data[key]]\\n        forbidden = {text_hash(r[\'ar\']) for r in heldout}\\n        eligible = []\\n        for r in pairs:\\n            if text_hash(r[\'ar\']) in forbidden:\\n                continue\\n            if any(token_jaccard(r[\'ar\'], v[\'ar\']) >= 0.8 for v in heldout):\\n                continue\\n            if len(mt_tokenizer(r[\'ar\'])[\'input_ids\']) > self.cfg[\'max_length\']:\\n                continue\\n            if len(mt_tokenizer(text_target=r[\'en\'])[\'input_ids\']) > self.cfg[\'max_length\']:\\n                continue\\n            eligible.append(r)\\n        from sentence_transformers import SentenceTransformer\\n        encoder = SentenceTransformer(self.cfg[\'quality_model\'], device=self.device)\\n        if not eligible:\\n            raise RuntimeError(\'No eligible back-translated pairs; inspect raw generation.\')\\n        ar_vectors = encoder.encode([r[\'ar\'] for r in eligible], normalize_embeddings=True, show_progress_bar=False)\\n        en_vectors = encoder.encode([r[\'en\'] for r in eligible], normalize_embeddings=True, show_progress_bar=False)\\n        seed_vectors = encoder.encode([r[\'en\'] for r in self.data[\'domain_train\']], normalize_embeddings=True, show_progress_bar=False)\\n        centroid = seed_vectors.mean(axis=0)\\n        centroid /= max(np.linalg.norm(centroid), 1e-12)\\n        bilingual = (ar_vectors * en_vectors).sum(axis=1)\\n        relevance = en_vectors @ centroid\\n        for i, row in enumerate(eligible):\\n            row.update({\'bilingual_similarity\': float(bilingual[i]), \'domain_similarity\': float(relevance[i]),\\n                        \'quality_score\': float(0.7*bilingual[i] + 0.3*relevance[i])})\\n        del encoder\\n        free_memory()\\n        k = min(self.cfg[\'synthetic_pairs\'], math.floor(len(eligible)*self.cfg[\'quality_keep_fraction\']))\\n        if k < self.cfg[\'minimum_synthetic_pairs\']:\\n            raise RuntimeError(f\'Only {len(eligible)} eligible pairs, matched experiment size {k}. \'\\n                               \'Increase prompt_count/sequences_per_prompt in a NEW run; no results fabricated.\')\\n        random_pool = random.Random(self.seed).sample(eligible, k)\\n        ranked_pool = sorted(eligible, key=lambda r: r[\'quality_score\'], reverse=True)[:k]\\n        self.pools = {\'synthetic_random\': random_pool, \'synthetic_ranked\': ranked_pool}\\n        write_json(cache, self.pools)\\n        pd.DataFrame(eligible).to_csv(self.root / \'quality_audit.csv\', index=False)\\n        write_json(self.root / \'quality_protocol.json\', {\'eligible_pairs\': len(eligible), \'matched_k\': k,\\n                   \'scoring\': \'0.7 cross-lingual cosine + 0.3 cosine to training English centroid\',\\n                   \'note\': \'Heuristic proxy, not a guarantee of semantic correctness. No dev/test reference used in ranking.\'})\\n        return pd.DataFrame([{\'pool\': key, \'pairs\': len(value)} for key, value in self.pools.items()])\\n\\n    def _tokenized(self, rows, tokenizer):\\n        dataset = Dataset.from_list([{\'ar\': r[\'ar\'], \'en\': r[\'en\']} for r in rows])\\n        def encode(batch):\\n            inputs = tokenizer(batch[\'ar\'], max_length=self.cfg[\'max_length\'], truncation=True)\\n            inputs[\'labels\'] = tokenizer(text_target=batch[\'en\'], max_length=self.cfg[\'max_length\'], truncation=True)[\'input_ids\']\\n            return inputs\\n        return dataset.map(encode, batched=True, remove_columns=dataset.column_names)\\n\\n    def train_variant(self, name, domain_rows):\\n        directory = self.root / name\\n        marker = directory / \'complete.json\'\\n        if marker.exists():\\n            return json.loads(marker.read_text())\\n        directory.mkdir(exist_ok=True)\\n        set_seed(self.seed)\\n        tokenizer = AutoTokenizer.from_pretrained(self.cfg[\'mt_model\'])\\n        model = AutoModelForSeq2SeqLM.from_pretrained(self.cfg[\'mt_model\']).to(self.device)\\n        self._record_model(self.cfg[\'mt_model\'], model)\\n        # A finite sampled schedule gives all adaptations identical update/example budgets.\\n        rng = random.Random(self.seed)\\n        draws = self.cfg[\'train_steps\'] * self.cfg[\'batch_size\'] * self.cfg[\'gradient_accumulation\']\\n        mixed = []\\n        domain_draws = 0\\n        for _ in range(draws):\\n            use_domain = rng.random() < self.cfg[\'in_domain_probability\']\\n            mixed.append(rng.choice(domain_rows if use_domain else self.data[\'generic_train\']))\\n            domain_draws += int(use_domain)\\n        write_json(directory / \'sampling.json\', {\'examples\': draws, \'domain_draws\': domain_draws,\\n                   \'domain_probability\': self.cfg[\'in_domain_probability\'], \'unique_domain_pool\': len(domain_rows),\\n                   \'unique_generic_pool\': len(self.data[\'generic_train\'])})\\n        training = self._tokenized(mixed, tokenizer)\\n        write_json(directory / \'token_budget.json\', {\\n            \'source_tokens_in_sampled_schedule\': sum(len(x) for x in training[\'input_ids\']),\\n            \'target_tokens_in_sampled_schedule\': sum(len(x) for x in training[\'labels\']),\\n            \'note\': \'Same updates and examples, but variable-length text means token budgets can differ.\'})\\n        development = self._tokenized(self.data[\'domain_dev\'], tokenizer)\\n        args = Seq2SeqTrainingArguments(\\n            output_dir=str(directory / \'checkpoints\'), max_steps=self.cfg[\'train_steps\'],\\n            learning_rate=self.cfg[\'learning_rate\'], per_device_train_batch_size=self.cfg[\'batch_size\'],\\n            per_device_eval_batch_size=self.cfg[\'batch_size\'], gradient_accumulation_steps=self.cfg[\'gradient_accumulation\'],\\n            eval_strategy=\'steps\', eval_steps=self.cfg[\'eval_steps\'], save_strategy=\'steps\', save_steps=self.cfg[\'eval_steps\'],\\n            save_total_limit=2, load_best_model_at_end=True, metric_for_best_model=\'eval_loss\', greater_is_better=False,\\n            predict_with_generate=False, logging_steps=25, report_to=\'none\', seed=self.seed, data_seed=self.seed,\\n            fp16=False, dataloader_num_workers=0, warmup_ratio=0.05, disable_tqdm=True)\\n        trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=training, eval_dataset=development,\\n                                data_collator=DataCollatorForSeq2Seq(tokenizer, model=model), processing_class=tokenizer)\\n        checkpoints = sorted((directory / \'checkpoints\').glob(\'checkpoint-*\'), key=lambda p: int(p.name.split(\'-\')[-1]))\\n        torch.cuda.synchronize()\\n        started = time.perf_counter()\\n        result = trainer.train(resume_from_checkpoint=str(checkpoints[-1]) if checkpoints else None)\\n        torch.cuda.synchronize()\\n        trainer.save_model(str(directory / \'model\'))\\n        tokenizer.save_pretrained(directory / \'model\')\\n        pd.DataFrame(trainer.state.log_history).to_csv(directory / \'training_history.csv\', index=False)\\n        summary = {\'variant\': name, \'best_checkpoint\': trainer.state.best_model_checkpoint,\\n                   \'best_dev_loss\': trainer.state.best_metric, \'steps\': trainer.state.global_step,\\n                   \'this_session_seconds\': time.perf_counter()-started, \'train_metrics\': result.metrics}\\n        write_json(marker, summary)\\n        del trainer, model\\n        free_memory()\\n        return summary\\n\\n    def train_all(self):\\n        variants = [(\'B_authentic_mix\', self.data[\'domain_train\']),\\n                    (\'C_synthetic_random\', self.pools[\'synthetic_random\']),\\n                    (\'D_synthetic_ranked\', self.pools[\'synthetic_ranked\'])]\\n        summaries = []\\n        for name, rows in variants:\\n            print(\'Training\', name, flush=True)\\n            summaries.append(self.train_variant(name, rows))\\n        return pd.DataFrame(summaries)\\n\\n    def average_checkpoint(self):\\n        path = self.root / \'E_ranked_average\'\\n        if (path / \'complete.json\').exists():\\n            return str(path / \'model\')\\n        model = AutoModelForSeq2SeqLM.from_pretrained(self.cfg[\'mt_model\'])\\n        adapted = AutoModelForSeq2SeqLM.from_pretrained(self.root / \'D_synthetic_ranked\' / \'model\')\\n        left, right = dict(model.named_parameters()), dict(adapted.named_parameters())\\n        assert left.keys() == right.keys()\\n        with torch.no_grad():\\n            for name in left:\\n                assert left[name].shape == right[name].shape\\n                left[name].mul_(0.5).add_(right[name], alpha=0.5)\\n        path.mkdir(exist_ok=True)\\n        model.save_pretrained(path / \'model\')\\n        AutoTokenizer.from_pretrained(self.cfg[\'mt_model\']).save_pretrained(path / \'model\')\\n        write_json(path / \'complete.json\', {\'alpha\': 0.5, \'parents\': [\'A_baseline\', \'D_synthetic_ranked\'],\\n                                         \'note\': \'Fixed coefficient, no test-set tuning.\'})\\n        del model, adapted\\n        free_memory()\\n        return str(path / \'model\')\\n\\n    def evaluate_all(self):\\n        models = {\'A_baseline\': self.cfg[\'mt_model\']}\\n        models.update({name: str(self.root / name / \'model\') for name in\\n                       [\'B_authentic_mix\', \'C_synthetic_random\', \'D_synthetic_ranked\', \'E_ranked_average\']})\\n        rows = []\\n        for name, checkpoint in models.items():\\n            pending = [split for split in (\'domain_dev\', \'generic_dev\', \'domain_test\', \'generic_test\')\\n                       if not (self.root / f\'{name}_{split}_scores.json\').exists()]\\n            if pending:\\n                model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(self.device).eval()\\n                tok = AutoTokenizer.from_pretrained(checkpoint)\\n                # Warm up with training text; no test outcome influences any configuration.\\n                self._translate(model, tok, [self.data[\'domain_train\'][0][\'ar\']])\\n                for split in pending:\\n                    data = self.data[split]\\n                    torch.cuda.synchronize()\\n                    started = time.perf_counter()\\n                    predictions = self._translate(model, tok, [r[\'ar\'] for r in data])\\n                    torch.cuda.synchronize()\\n                    seconds = time.perf_counter()-started\\n                    refs = [r[\'en\'] for r in data]\\n                    bleu, chrf, ter = BLEU(), CHRF(word_order=2), TER()\\n                    score = {\'model\': name, \'split\': split, \'n\': len(data),\\n                             \'BLEU\': bleu.corpus_score(predictions, [refs]).score,\\n                             \'chrF++\': chrf.corpus_score(predictions, [refs]).score,\\n                             \'TER\': ter.corpus_score(predictions, [refs]).score, \'seconds\': seconds,\\n                             \'BLEU_signature\': str(bleu.get_signature()), \'chrF_signature\': str(chrf.get_signature())}\\n                    frame = pd.DataFrame({\'id\': [r[\'id\'] for r in data], \'source_ar\': [r[\'ar\'] for r in data],\\n                                          \'reference_en\': refs, \'prediction_en\': predictions})\\n                    frame.to_csv(self.root / f\'{name}_{split}_predictions.csv\', index=False)\\n                    write_json(self.root / f\'{name}_{split}_scores.json\', score)\\n                    print(name, split, score[\'BLEU\'], score[\'chrF++\'], flush=True)\\n                del model\\n                free_memory()\\n            for split in (\'domain_dev\', \'generic_dev\', \'domain_test\', \'generic_test\'):\\n                rows.append(json.loads((self.root / f\'{name}_{split}_scores.json\').read_text()))\\n        results = pd.DataFrame(rows)\\n        results.to_csv(self.root / \'results.csv\', index=False)\\n        return results\\n\\n    def bootstrap_improvement(self, repeats=300):\\n        baseline = pd.read_csv(self.root / \'C_synthetic_random_domain_test_predictions.csv\')\\n        improved = pd.read_csv(self.root / \'D_synthetic_ranked_domain_test_predictions.csv\')\\n        assert baseline.id.tolist() == improved.id.tolist()\\n        docs = [r[\'document\'] for r in self.data[\'domain_test\']]\\n        groups = sorted(set(docs))\\n        rng = np.random.default_rng(self.seed)\\n        metric, differences = CHRF(word_order=2), []\\n        # Cluster bootstrap respects within-document correlation.\\n        for _ in range(repeats):\\n            sampled = rng.choice(groups, size=len(groups), replace=True)\\n            ids = [i for group in sampled for i, doc in enumerate(docs) if doc == group]\\n            refs = baseline.reference_en.iloc[ids].tolist()\\n            c = metric.corpus_score(baseline.prediction_en.iloc[ids].tolist(), [refs]).score\\n            d = metric.corpus_score(improved.prediction_en.iloc[ids].tolist(), [refs]).score\\n            differences.append(d-c)\\n        interval = np.percentile(differences, [2.5, 97.5])\\n        summary = {\'comparison\': \'D minus C, domain test chrF++\', \'repeats\': repeats,\\n                   \'document_clusters\': len(groups), \'ci95_low\': interval[0], \'ci95_high\': interval[1],\\n                   \'caution\': \'Pilot cluster bootstrap, few documents and one training seed; not robust evidence across runs.\'}\\n        write_json(self.root / \'bootstrap.json\', summary)\\n        return summary\\n", "configs/pilot.json": "{\\n  \\"seed\\": 42,\\n  \\"mt_model\\": \\"Helsinki-NLP/opus-mt-ar-en\\",\\n  \\"reverse_model\\": \\"Helsinki-NLP/opus-mt-en-ar\\",\\n  \\"lm_model\\": \\"EleutherAI/pythia-410m\\",\\n  \\"quality_model\\": \\"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2\\",\\n  \\"tico_url\\": \\"https://tico-19.github.io/data/tico19-testset.zip\\",\\n  \\"authentic_pairs\\": 200,\\n  \\"domain_dev\\": 100,\\n  \\"domain_test\\": 200,\\n  \\"generic_train\\": 3600,\\n  \\"generic_dev\\": 100,\\n  \\"generic_test\\": 200,\\n  \\"prompt_count\\": 120,\\n  \\"sequences_per_prompt\\": 3,\\n  \\"lm_new_tokens\\": 128,\\n  \\"synthetic_pairs\\": 400,\\n  \\"minimum_synthetic_pairs\\": 60,\\n  \\"max_length\\": 128,\\n  \\"max_new_tokens\\": 128,\\n  \\"batch_size\\": 4,\\n  \\"gradient_accumulation\\": 4,\\n  \\"train_steps\\": 300,\\n  \\"eval_steps\\": 100,\\n  \\"learning_rate\\": 0.00002,\\n  \\"in_domain_probability\\": 0.9,\\n  \\"quality_keep_fraction\\": 0.5\\n}\\n", "tests/test_data.py": "import unittest\\nfrom domainmt.data import (assert_disjoint, clean_pairs, continuation_sentences, normalize,\\n                           split_documents, token_jaccard)\\n\\n\\nclass DataTests(unittest.TestCase):\\n    def test_clean(self):\\n        pair = {\'ar\': \'هذا مثال عربي واضح\', \'en\': \'This is an English example.\'}\\n        rows = [pair, pair.copy(), {\'ar\': \' \', \'en\': \'hello\'},\\n                {\'ar\': \'English\', \'en\': \'English\'}, {\'ar\': None, \'en\': \'x\'}]\\n        self.assertEqual(clean_pairs(rows), [pair])\\n\\n    def test_disjoint(self):\\n        with self.assertRaises(AssertionError):\\n            assert_disjoint({\'a\': [{\'ar\': \'نص\', \'en\': \' A sentence \'}],\\n                             \'b\': [{\'ar\': \'آخر\', \'en\': \'a sentence\'}]})\\n\\n    def test_document_split(self):\\n        rows = [{\'document\': str(i), \'ar\': f\'نص {i}\', \'en\': f\'English {i}\'} for i in range(30)]\\n        first = split_documents(rows)\\n        self.assertEqual(first, split_documents(rows))\\n        self.assertEqual(sum(map(len, first.values())), 30)\\n        assert_disjoint(first)\\n        sets = [{r[\'document\'] for r in rows} for rows in first.values()]\\n        self.assertTrue(sets[0].isdisjoint(sets[1]))\\n\\n    def test_continuations(self):\\n        text = \'People should wash their hands with soap. This trailing fragment has no final stop\'\\n        self.assertEqual(continuation_sentences(text), [\'People should wash their hands with soap.\'])\\n\\n    def test_normalization(self):\\n        self.assertEqual(normalize(\'  Test\\\\nCASE \'), \'test case\')\\n        self.assertEqual(token_jaccard(\'a b c\', \'c b a\'), 1)\\n        self.assertEqual(token_jaccard(\'\', \'\'), 0)\\n\\n\\nif __name__ == \'__main__\':\\n    unittest.main()\\n"}')
PROJECT = Path("/kaggle/working/tmg301_source") if Path("/kaggle").exists() else Path.cwd() / "tmg301_source"
for relative, content in EMBEDDED_FILES.items():
    path = PROJECT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
sys.path.insert(0, str(PROJECT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from datasets import disable_progress_bars
from transformers.utils import logging
disable_progress_bars()
logging.disable_progress_bar()
pd.set_option("display.max_colwidth", 180)
from domainmt.experiment import Experiment
config = json.loads((PROJECT / "configs/pilot.json").read_text())
display(pd.Series(config, name="setting").to_frame())


## 1. Fixed protocol and environment
The pilot runs all five systems: A pretrained baseline; B authentic mixed fine-tuning; C randomly sampled synthetic mixed fine-tuning; D quality-ranked synthetic mixed fine-tuning; E parameter averaging of D and A with fixed weight 0.5. Each adaptation starts independently from A. C and D use equal synthetic-pool sizes and identical training update/example budgets; actual token counts are exported because sequence lengths differ. Training samples in-domain examples with probability 0.9 and generic examples with probability 0.1. This is the sampling probability, not the stored corpus-size ratio.

Checkpoints for B/C/D are selected by **domain dev loss**, never test metrics. The experiment uses one seed and small held-out sets; it can support a pilot result, not a definitive significance claim. An improvement may fail or harm general-domain performance; negative results must be reported.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(PROJECT / "tests"), "-v"],
               env={**os.environ, "PYTHONPATH": str(PROJECT / "src")}, check=True)
output_root = Path("/kaggle/working/domain_mt_runs") if Path("/kaggle").exists() else Path.cwd() / "domain_mt_runs"
experiment = Experiment(config, output_root)
display(pd.Series(experiment.versions, name="version").to_frame())


## 2. Authentic data and leakage controls
TICO-19 is downloaded from the official archive. Both official partitions are combined and **repartitioned by document ID** before model use; these are custom research splits, not official benchmark scores. Cross-split exact source/target duplicates are removed and deterministic subsets are drawn. Only 200 training pairs provide domain prompts and authentic training data. Test references are excluded from generation/training; they are used only for mechanical overlap exclusion and final scoring, never as generation prompts or quality-ranking features.

OPUS-100 Arabic-English supplies generic replay and separate generic dev/test samples. Exact overlaps with TICO and other sampled partitions are excluded. Pretraining contamination cannot be ruled out for public checkpoints. Raw strings and provenance are retained. No dataset redistribution or blanket license claim is made.

In [ ]:
display(experiment.prepare_data())
display(pd.DataFrame(experiment.data["domain_train"]).head(5))
print("Saved document partitions and all data subsets:", experiment.root)


## 3. Generate English target-side domain text
English training sentences prompt a small English causal LM using top-k 50, top-p 0.95 and three sampled continuations per prompt. Prompt tokens are removed before collecting complete sentences. Basic filtering removes duplicates, non-English text, incomplete fragments, and exact/high-Jaccard copies of seed or evaluation sentences. These controls reduce memorization risk but cannot prove absence of all semantic overlap. Prompt provenance and raw continuations are saved; progress is checkpointed per prompt. A small LM may drift off-domain, and synthetic health statements must not be treated as medical advice.

In [ ]:
targets = experiment.generate_targets()
print("Accepted English targets:", len(targets))
display(targets.head(10))


## 4. Back-translation and matched quality-filtering ablation
Translate generated English into Arabic with the reverse model (beam 5). The resulting training direction is `(synthetic Arabic -> generated English)`. Reject wrong-script/duplicate/overlength pairs and held-out near-copies. Rank remaining pairs by **0.7 bilingual embedding cosine + 0.3 domain-centroid cosine**; the domain centroid uses authentic training English only. The coefficients are fixed a priori. This is a quality proxy, not a semantic guarantee.

C randomly draws K pairs; D takes the top K from the same eligible pool, where K is at most 400 and at most half the pool. Both have the same K. The minimum-size guard stops an uninformative run rather than fabricating results. If it fails, inspect generation and increase the generation budget in a new run before any final-test evaluation. This recipe is a methodological extension, not the paper's original 0.45 semantic threshold.

In [ ]:
display(experiment.backtranslate_and_rank())
quality = pd.read_csv(experiment.root / "quality_audit.csv")
display(quality[["en", "ar", "bilingual_similarity", "domain_similarity", "quality_score"]].head(8))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(quality.quality_score, bins=25, color="#287f8e", edgecolor="white")
ax.set(title="Synthetic quality-score distribution", xlabel="Heuristic score", ylabel="Pairs")
fig.tight_layout()
fig.savefig(experiment.root / "quality_scores.png", dpi=150)
plt.show()


## 5. Independent mixed fine-tuning and checkpoint averaging
Each of B, C and D starts from the same pretrained weights and uses 300 optimizer steps, batch size 4 with accumulation 4, learning rate 2e-5 and maximum source/target length 128. Validation and checkpointing occur every 100 steps. Full-precision training is chosen for stability; only one GPU is used even when two are available. Interrupted training can resume from a checkpoint in the same run folder. Completed variants are cached.

E averages corresponding parameters of A and D at fixed 0.5/0.5 weight. This is weight averaging between compatible models, not a prediction ensemble and not averaging unrelated architectures. No test score determines the coefficient.

In [ ]:
training = experiment.train_all()
display(training)
print("Averaged model:", experiment.average_checkpoint())
fig, ax = plt.subplots(figsize=(9, 4))
for name in ["B_authentic_mix", "C_synthetic_random", "D_synthetic_ranked"]:
    history = pd.read_csv(experiment.root / name / "training_history.csv")
    if "eval_loss" in history:
        points = history.dropna(subset=["eval_loss"])
        ax.plot(points.step, points.eval_loss, marker="o", label=name)
ax.set(title="Domain-development loss (checkpoint selection)", xlabel="Optimizer step", ylabel="Cross-entropy loss")
ax.legend()
fig.tight_layout()
fig.savefig(experiment.root / "dev_loss.png", dpi=150)
plt.show()


## 6. Frozen evaluation on both domains
All five predeclared systems are evaluated on the same domain and generic sets using beam 4. Predictions and scores are cached and reused; inspecting test scores must not lead to tuning and retesting on these same sets. We report corpus BLEU (13a), chrF++ (word order 2), TER, sample counts and synchronized GPU runtime. **BLEU here is not the paper's spBLEU**, and neither metric nor split is identical to the original table. COMET and native-Arabic expert human evaluation are not included in this pilot. Downloads/training are excluded from generation timing.

In [ ]:
results = experiment.evaluate_all()
display(results[["model", "split", "n", "BLEU", "chrF++", "TER", "seconds"]])
test_results = results[results.split.isin(["domain_test", "generic_test"])]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, split in zip(axes, ["domain_test", "generic_test"]):
    frame = test_results[test_results.split == split].set_index("model")
    ax.bar([name[:1] for name in frame.index], frame["chrF++"], color=["#65717a", "#3879a6", "#259783", "#b65f74", "#9b843e"])
    ax.set(title=split, xlabel="System A-E", ylabel="Corpus chrF++", ylim=(0, 100))
fig.tight_layout()
fig.savefig(experiment.root / "test_comparison.png", dpi=150)
plt.show()
display(pd.Series(experiment.bootstrap_improvement(), name="result").to_frame())


## 7. Inspect actual errors, not invented explanations
The following table joins real outputs by sentence ID. Low overlap is only a screening signal. Compare terminology, negation, quantities, omissions and hallucinations against the Arabic source and English reference. Consult a competent Arabic speaker for a defensible human evaluation. The exported review sheet intentionally leaves human judgements empty: code cannot truthfully claim that these have already been assessed.

In [ ]:
from sacrebleu.metrics import CHRF
review = pd.read_csv(experiment.root / "A_baseline_domain_test_predictions.csv").rename(columns={"prediction_en": "A_prediction"})
for name in ["B_authentic_mix", "C_synthetic_random", "D_synthetic_ranked", "E_ranked_average"]:
    frame = pd.read_csv(experiment.root / f"{name}_domain_test_predictions.csv")
    review = review.merge(frame[["id", "prediction_en"]].rename(columns={"prediction_en": name[0]+"_prediction"}), on="id", validate="one_to_one")
metric = CHRF(word_order=2)
review["D_sentence_chrF++"] = [metric.sentence_score(p, [r]).score for p, r in zip(review.D_prediction, review.reference_en)]
review = review.sort_values("D_sentence_chrF++")
review["human_error_category"] = ""
review["human_explanation"] = ""
review.to_csv(experiment.root / "human_review.csv", index=False)
display(review.head(10))


## 8. Measured conclusion and artifacts
The summary below reports whether ranking helped in this run, including degradation if present. It does not promise the paper's gains. Before presenting the project, add a qualitative discussion using actual outputs and repeat important comparisons with more training seeds when resources permit. Do not use the current test set for further model selection.

In [ ]:
lookup = results.set_index(["model", "split"])
c = lookup.loc[("C_synthetic_random", "domain_test"), "chrF++"]
d = lookup.loc[("D_synthetic_ranked", "domain_test"), "chrF++"]
a_general = lookup.loc[("A_baseline", "generic_test"), "chrF++"]
d_general = lookup.loc[("D_synthetic_ranked", "generic_test"), "chrF++"]
summary = f"Quality-ranked D minus random-synthetic C: {d-c:+.3f} domain-test chrF++ points. D minus pretrained A: {d_general-a_general:+.3f} generic-test chrF++ points. These are measured pilot results, not guaranteed improvements. All adaptations use the same update count; token budgets, source data and pretrained models differ from the original paper. Human evaluation remains pending."
display(Markdown(summary))
(experiment.root / "measured_summary.txt").write_text(summary)
assert len(results) == 20 and results[["BLEU", "chrF++", "TER"]].notna().all().all()
assert len(experiment.pools["synthetic_random"]) == len(experiment.pools["synthetic_ranked"])
import zipfile
archive = experiment.root / "reports_and_predictions.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in experiment.root.rglob("*"):
        if path.is_file() and path.suffix in [".csv", ".json", ".png", ".txt"] and "checkpoints" not in path.parts and "model" not in path.parts:
            z.write(path, path.relative_to(experiment.root))
print("Reports:", archive)
print("Models: B/C/D/E subfolders under", experiment.root)
print("Save Version with outputs and download this executed notebook. No model weights or dataset files are pushed to GitHub.")


## References and limitations
- Moslem et al. (2022): https://aclanthology.org/2022.amta-research.2/
- Authors' implementation: https://github.com/ymoslem/MT-LM
- TICO-19: https://tico-19.github.io/testset.html
- Generic corpus: https://huggingface.co/datasets/Helsinki-NLP/opus-100
- MT: https://huggingface.co/Helsinki-NLP/opus-mt-ar-en and https://huggingface.co/Helsinki-NLP/opus-mt-en-ar
- English LM: https://huggingface.co/EleutherAI/pythia-410m
- Quality encoder: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Metrics: https://github.com/mjpost/sacrebleu

The code is an independently written, AI-assisted implementation of the method; it does not copy the author's implementation. Cite the paper and disclose assistance according to course rules. Public-data licensing, potential model-pretraining overlap, limited compute, small sample size, one seed, heuristic filtering and lack of native-speaker evaluation constrain the conclusions. Generated health-domain material is experimental synthetic text and must not be used as medical advice.